# PBL5 - 2-Stage Damage Detection Pipeline

**Stage 1**: YOLOv8 binary detector (1 class = 'damage') -> tim VUNG hu hai
**Stage 2**: YOLOv8 classifier (8 class) -> phan loai LOAI hu hai tren crop

## Tai sao 2-stage?
- 1-stage 8-class bi domain shift (val 84% -> test 6%) vi model phai vua tim vua phan loai cung luc
- 2-stage tach dieu: Stage 1 (binary, de generalize) -> Stage 2 (classify tren crop, mat context global it bi anh huong)

## Cach dung
1. Add Data zip `data_26_04_2stage_*.zip` (hoac folder dataset_2stage da extract)
2. Settings: GPU T4 x2, Internet ON
3. Chay tuan tu — co the chay tach roi tung stage neu can

## Thoi gian uoc tinh (T4 x2)
- Stage 1 (binary, 200 ep): ~2.5h
- Stage 2 (classify, 100 ep): ~1h
- Combined eval: ~5 phut

## 0. Setup environment

In [ ]:
import os, sys, zipfile, shutil, glob, json
from pathlib import Path
from datetime import datetime
from collections import Counter

# Tat noise tu TF/absl/cuDNN tren Kaggle
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['GRPC_VERBOSITY'] = 'ERROR'
os.environ['ABSL_LOGGING_VERBOSITY'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
import warnings; warnings.filterwarnings('ignore')

!pip uninstall -y ray 2>/dev/null | tail -1
!pip install -q -U ultralytics==8.3.39 timm

import torch
print('Torch:', torch.__version__, '| CUDA:', torch.cuda.is_available(), '| GPUs:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    free, tot = torch.cuda.mem_get_info(i)
    print(f'  GPU{i}: {torch.cuda.get_device_name(i)}  ({tot/1e9:.1f} GB)')

## 1. Tim/extract dataset 2-stage

In [ ]:
WORK = Path('/kaggle/working')
INPUT = Path('/kaggle/input')

def find_2stage_root(base: Path):
    """Tim folder co cau truc stage1/images/train + stage2/train/<class>"""
    for cand in base.rglob('stage1'):
        if (cand / 'images' / 'train').is_dir() and (cand.parent / 'stage2' / 'train').is_dir():
            return cand.parent
    return None

DATA_ROOT = find_2stage_root(INPUT)

if DATA_ROOT is not None:
    print(f'Found extracted: {DATA_ROOT}')
else:
    zips = list(INPUT.rglob('data_26_04_2stage_*.zip')) or list(INPUT.rglob('*2stage*.zip'))
    assert zips, 'Khong tim thay dataset 2-stage trong /kaggle/input/'
    ZIP = zips[0]
    print(f'Extracting {ZIP} ({ZIP.stat().st_size/1e6:.1f} MB)...')
    extract_root = WORK / 'data_extracted'
    if extract_root.exists():
        shutil.rmtree(extract_root)
    extract_root.mkdir(parents=True)
    with zipfile.ZipFile(ZIP) as zf:
        zf.extractall(extract_root)
    DATA_ROOT = find_2stage_root(extract_root)
    assert DATA_ROOT is not None, f'Sau extract khong tim thay stage1/stage2 trong {extract_root}'
    print(f'Extracted to: {DATA_ROOT}')

STAGE1_DIR = DATA_ROOT / 'stage1'
STAGE2_DIR = DATA_ROOT / 'stage2'

# Verify
for s in ('train', 'val', 'test'):
    n_s1 = len(list((STAGE1_DIR / 'images' / s).iterdir()))
    n_s2 = sum(len(list(d.iterdir())) for d in (STAGE2_DIR / s).iterdir() if d.is_dir())
    print(f'  {s:5s}: stage1={n_s1:5d} images  | stage2={n_s2:5d} crops')

In [ ]:
# Stage 1 yaml: tro path: ve /kaggle/input (read-only OK vi YOLO chi DOC)
import yaml
STAGE1_YAML = WORK / 'stage1.yaml'
yaml.safe_dump({
    'path': STAGE1_DIR.as_posix(),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': 1,
    'names': ['damage'],
}, open(STAGE1_YAML, 'w'), sort_keys=False)
print(STAGE1_YAML.read_text())

## 2. Class distribution Stage 2

In [ ]:
CLASS_NAMES = ['material_loss', 'peel', 'scratch', 'fold',
               'writing_marks', 'dirt', 'staning', 'burn_marks']

print(f'  {"class":<17} {"train":>8} {"val":>8} {"test":>8}')
print(f'  {"-"*17} {"-"*8} {"-"*8} {"-"*8}')
for cname in CLASS_NAMES:
    counts = []
    for split in ('train', 'val', 'test'):
        d = STAGE2_DIR / split / cname
        counts.append(len(list(d.iterdir())) if d.is_dir() else 0)
    print(f'  {cname:<17} {counts[0]:>8} {counts[1]:>8} {counts[2]:>8}')

## 3. STAGE 1 - Train YOLOv8 binary detector

**Muc tieu**: Tim VUNG hu hai. Khong can phan biet loai.

**Bi quyet:**
- Augment **manh hon** (vi binary -> ko sai class) -> generalize cao
- imgsz 832 (bat damage nho)
- yolov8m -> tradeoff toc do/accuracy

In [ ]:
# ====== STAGE 1 CONFIG ======
S1_MODEL    = 'yolov8m.pt'
S1_EPOCHS   = 200
S1_BATCH    = 12
S1_IMGSZ    = 832
S1_PATIENCE = 40
S1_LR0      = 0.001
S1_DEVICE   = 0

if torch.cuda.device_count() >= 2:
    S1_DEVICE = '0,1'
    S1_BATCH = S1_BATCH * 2
    print(f'Dual GPU: device={S1_DEVICE}, batch={S1_BATCH}')

S1_RUN = f"stage1_{datetime.now().strftime('%m%d_%H%M')}_{S1_MODEL.replace('.pt','')}"
S1_PROJECT = '/kaggle/working/runs/stage1'
print('Stage 1 run:', S1_RUN)

In [ ]:
from ultralytics import YOLO

model_s1 = YOLO(S1_MODEL)

# Disable broken raytune callback
for hook in list(model_s1.callbacks.keys()):
    model_s1.callbacks[hook] = [
        cb for cb in model_s1.callbacks[hook]
        if 'raytune' not in getattr(cb, '__module__', '')
    ]

model_s1.train(
    data=str(STAGE1_YAML),
    epochs=S1_EPOCHS,
    batch=S1_BATCH,
    imgsz=S1_IMGSZ,
    device=S1_DEVICE,
    name=S1_RUN,
    project=S1_PROJECT,
    exist_ok=True,
    patience=S1_PATIENCE,
    workers=4,
    seed=42,
    deterministic=True,

    # Optimizer
    optimizer='AdamW',
    lr0=S1_LR0,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=5,
    momentum=0.937,
    cos_lr=True,

    # AUGMENTATION manh hon (binary -> khong sai class)
    hsv_h=0.015,            # +50% so 1-stage
    hsv_s=0.50,
    hsv_v=0.40,
    degrees=10.0,
    translate=0.10,
    scale=0.40,             # +33% scale variation
    shear=3.0,
    perspective=0.0,
    flipud=0.20,            # binary -> flip up tu do hon
    fliplr=0.50,
    mosaic=0.80,            # +60% mosaic vi binary
    mixup=0.15,
    copy_paste=0.30,
    erasing=0.30,

    # Loss
    cls=0.5,                # binary -> cls weight nho
    box=7.5,
    dfl=1.5,

    val=True, plots=True, save=True,
    save_period=25, close_mosaic=15, amp=True,
    cache=False, rect=False, nbs=64,
)

S1_BEST = Path(S1_PROJECT) / S1_RUN / 'weights' / 'best.pt'
print(f'\n[Stage 1 done] best: {S1_BEST}')

### 3.1 Validate Stage 1 tren val + test

In [ ]:
model_s1 = YOLO(str(S1_BEST))
for split in ('val', 'test'):
    print(f'\n=== Stage 1 - {split.upper()} ===')
    m = model_s1.val(
        data=str(STAGE1_YAML), split=split,
        batch=S1_BATCH, imgsz=S1_IMGSZ, device=S1_DEVICE,
        plots=True, save_json=True, conf=0.001, iou=0.6,
        augment=True,           # TTA
        name=f's1_{split}_{S1_RUN}', project=S1_PROJECT, exist_ok=True,
    )
    print(f'  mAP50:    {m.box.map50:.4f}')
    print(f'  mAP50-95: {m.box.map:.4f}')
    print(f'  P:        {m.box.mp:.4f}')
    print(f'  R:        {m.box.mr:.4f}')

## 4. STAGE 2 - Train YOLOv8 classifier (8-class)

**Muc tieu**: Phan loai loai hu hai tren crop (256x256, train 224x224)

**Bi quyet:**
- Crop nho -> ResNet/EfficientNet-style features hieu qua
- yolov8s-cls la mac dinh tot, yolov8m-cls cho accuracy cao hon
- Augment vua phai (crop nho -> de bi distort)

In [ ]:
# ====== STAGE 2 CONFIG ======
S2_MODEL    = 'yolov8m-cls.pt'   # m = mAP cao hon, s = nhanh hon
S2_EPOCHS   = 100
S2_BATCH    = 64                 # cls model nho hon, batch lon
S2_IMGSZ    = 224
S2_PATIENCE = 30
S2_LR0      = 0.001
S2_DEVICE   = 0

if torch.cuda.device_count() >= 2:
    S2_DEVICE = '0,1'
    S2_BATCH = S2_BATCH * 2

S2_RUN = f"stage2_{datetime.now().strftime('%m%d_%H%M')}_{S2_MODEL.replace('.pt','')}"
S2_PROJECT = '/kaggle/working/runs/stage2'
print('Stage 2 run:', S2_RUN)

In [ ]:
model_s2 = YOLO(S2_MODEL)

for hook in list(model_s2.callbacks.keys()):
    model_s2.callbacks[hook] = [
        cb for cb in model_s2.callbacks[hook]
        if 'raytune' not in getattr(cb, '__module__', '')
    ]

# Stage 2 dung folder structure: stage2/{train,val,test}/<class>/
model_s2.train(
    data=str(STAGE2_DIR),       # truyen folder root
    epochs=S2_EPOCHS,
    batch=S2_BATCH,
    imgsz=S2_IMGSZ,
    device=S2_DEVICE,
    name=S2_RUN,
    project=S2_PROJECT,
    exist_ok=True,
    patience=S2_PATIENCE,
    workers=4,
    seed=42,
    deterministic=True,

    # Optimizer
    optimizer='AdamW',
    lr0=S2_LR0,
    lrf=0.01,
    weight_decay=0.0001,
    warmup_epochs=3,
    momentum=0.937,
    cos_lr=True,

    # AUGMENTATION cho crop classification (vua phai)
    hsv_h=0.015, hsv_s=0.40, hsv_v=0.30,
    degrees=15.0,                # crop -> cho phep rotate them
    translate=0.05,              # crop nho -> translate it
    scale=0.20,                  # scale nho
    fliplr=0.50, flipud=0.10,
    erasing=0.25,                # random erasing mid
    auto_augment='randaugment',  # YOLO classify ho tro RandAugment

    # Misc
    val=True, plots=True, save=True,
    save_period=20, amp=True,
)

S2_BEST = Path(S2_PROJECT) / S2_RUN / 'weights' / 'best.pt'
print(f'\n[Stage 2 done] best: {S2_BEST}')

### 4.1 Validate Stage 2 standalone

In [ ]:
model_s2 = YOLO(str(S2_BEST))
for split in ('val', 'test'):
    print(f'\n=== Stage 2 standalone - {split.upper()} (top-1/top-5 accuracy tren crop) ===')
    m = model_s2.val(
        data=str(STAGE2_DIR), split=split,
        batch=S2_BATCH, imgsz=S2_IMGSZ, device=S2_DEVICE,
        plots=True,
        name=f's2_{split}_{S2_RUN}', project=S2_PROJECT, exist_ok=True,
    )
    print(f'  top-1 acc: {m.top1:.4f}')
    print(f'  top-5 acc: {m.top5:.4f}')

## 5. COMBINED PIPELINE - Stage 1 + Stage 2 tren test set

**Pipeline**: anh test -> Stage 1 detect bbox 'damage' -> crop tung bbox + pad 20% -> Stage 2 classify -> ket qua final

Tinh mAP@50 tren test set (giong YOLO val) sau khi gop bbox + class predicted.

In [ ]:
import numpy as np
from PIL import Image

# Load both models
model_s1 = YOLO(str(S1_BEST))
model_s2 = YOLO(str(S2_BEST))

# Stage 2 class index -> ten class (Ultralytics classify dung folder name as class)
S2_NAMES = model_s2.names if hasattr(model_s2, 'names') else CLASS_NAMES
if isinstance(S2_NAMES, dict):
    S2_NAMES = [S2_NAMES[i] for i in sorted(S2_NAMES)]
print('Stage 2 class names:', S2_NAMES)

# Map class name -> index theo CLASS_NAMES toan cuc
NAME_TO_GLOBAL = {n: i for i, n in enumerate(CLASS_NAMES)}

# Test images
TEST_IMG_DIR = STAGE1_DIR / 'images' / 'test'
TEST_LBL_DIR = STAGE1_DIR / 'labels' / 'test'
test_imgs = sorted(TEST_IMG_DIR.iterdir())
print(f'Test images: {len(test_imgs)}')

In [ ]:
def crop_with_pad(im, x1, y1, x2, y2, pad=0.20, size=224):
    """Crop bbox + padding 20% -> resize ve square."""
    W, H = im.size
    bw, bh = x2 - x1, y2 - y1
    px, py = bw * pad, bh * pad
    cx1 = max(0, int(x1 - px))
    cy1 = max(0, int(y1 - py))
    cx2 = min(W, int(x2 + px))
    cy2 = min(H, int(y2 + py))
    crop = im.crop((cx1, cy1, cx2, cy2))
    crop.thumbnail((size, size), Image.LANCZOS)
    bg = Image.new('RGB', (size, size), (128, 128, 128))
    bg.paste(crop, ((size - crop.width) // 2, (size - crop.height) // 2))
    return bg


def run_pipeline(img_path, conf_s1=0.10, iou_s1=0.5):
    """Stage 1 + Stage 2 chung thanh detect 8-class final."""
    # Stage 1: detect 'damage'
    r1 = model_s1.predict(img_path, conf=conf_s1, iou=iou_s1, imgsz=S1_IMGSZ,
                          device=S1_DEVICE, save=False, verbose=False)[0]
    if r1.boxes is None or len(r1.boxes) == 0:
        return [], [], []
    boxes = r1.boxes.xyxy.cpu().numpy()           # (N, 4)
    s1_confs = r1.boxes.conf.cpu().numpy()         # (N,)

    # Crop + Stage 2 classify
    with Image.open(img_path) as im:
        if im.mode != 'RGB':
            im = im.convert('RGB')
        crops = [crop_with_pad(im, *b, size=S2_IMGSZ) for b in boxes]
    r2_list = model_s2.predict(crops, imgsz=S2_IMGSZ, device=S2_DEVICE,
                               save=False, verbose=False)
    classes, s2_confs = [], []
    for r2 in r2_list:
        # Probs: vector len 8
        probs = r2.probs.data.cpu().numpy() if r2.probs is not None else None
        if probs is None:
            classes.append(-1); s2_confs.append(0.0)
            continue
        cls_idx = int(np.argmax(probs))
        cls_name = S2_NAMES[cls_idx]
        global_idx = NAME_TO_GLOBAL.get(cls_name, -1)
        classes.append(global_idx)
        s2_confs.append(float(probs[cls_idx]))

    # Combined confidence: s1 * s2 (de loc bot detection chac chan thap)
    final_conf = s1_confs * np.array(s2_confs)
    return boxes.tolist(), classes, final_conf.tolist()

# Quick test 1 anh
boxes, cls, confs = run_pipeline(test_imgs[0])
print(f'Sample {test_imgs[0].name}: {len(boxes)} detections')
for b, c, f in zip(boxes[:5], cls[:5], confs[:5]):
    nm = CLASS_NAMES[c] if c >= 0 else 'UNKNOWN'
    print(f'  bbox={[int(x) for x in b]}  class={nm}  conf={f:.3f}')

### 5.1 Compute mAP cua combined pipeline tren test set

In [ ]:
from tqdm import tqdm

def load_gt(stem):
    """Doc label file (YOLO 8-class), tra ve list (cls, x1, y1, x2, y2) PIXEL."""
    # Doc tu STAGE2 source dataset (yolo_dataset_clean tuong duong) — thuc te STAGE1 labels da bien thanh binary
    # -> can luu rieng GT 8-class. Workaround: dung label class indices tu Stage 2 folder mapping
    # Don gian: doc label file goc tu /kaggle/input dataset_train_2704 (neu co)
    pass  # see next cell

# Tim 8-class GT labels: Stage 1 da bi remap binary, can label goc.
# Cach 1: label cua Stage 1 -> binary (chi co class 0). KHONG dung de tinh per-class mAP.
# Cach 2: nguoc tu Stage 2 crops back to original images bbox.
# DON GIAN NHAT: re-load GT 8-class tu yolo_dataset_clean neu co san trong /kaggle/input

# Tim yolo_dataset_clean 8-class trong input
def find_yolo_clean(base):
    for cand in base.rglob('data.yaml'):
        try:
            cfg = yaml.safe_load(open(cand))
            if cfg.get('nc') == 8:
                return cand.parent
        except: pass
    return None

GT_ROOT = find_yolo_clean(INPUT)
if GT_ROOT is None:
    GT_ROOT = find_yolo_clean(WORK)
print(f'GT 8-class root: {GT_ROOT}')

if GT_ROOT is None:
    print('WARN: khong tim thay GT 8-class (yolo_dataset_clean). Bo qua mAP combined.')
else:
    GT_IMG_TEST = GT_ROOT / 'images' / 'test'
    GT_LBL_TEST = GT_ROOT / 'labels' / 'test'
    print(f'GT test: {len(list(GT_IMG_TEST.iterdir()))} images, {len(list(GT_LBL_TEST.iterdir()))} labels')

In [ ]:
# Tinh mAP@50 cua combined pipeline = tinh per-class IoU vs GT
def yolo_to_xyxy(cx, cy, w, h, W, H):
    return ((cx - w/2)*W, (cy - h/2)*H, (cx + w/2)*W, (cy + h/2)*H)

def iou(boxA, boxB):
    xA = max(boxA[0], boxB[0]); yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2]); yB = min(boxA[3], boxB[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    a1 = (boxA[2]-boxA[0]) * (boxA[3]-boxA[1])
    a2 = (boxB[2]-boxB[0]) * (boxB[3]-boxB[1])
    return inter / (a1 + a2 - inter + 1e-9)

if GT_ROOT is not None:
    # Tap predictions + GT cho moi class
    preds_per_class = {c: [] for c in range(8)}     # (conf, is_tp)
    n_gt_per_class = Counter()

    for img_path in tqdm(test_imgs, desc='Combined eval'):
        # Run pipeline
        boxes, classes, confs = run_pipeline(img_path, conf_s1=0.10)

        # GT load
        gt_lbl = GT_LBL_TEST / f'{img_path.stem}.txt'
        if not gt_lbl.exists():
            continue
        with Image.open(img_path) as im:
            W, H = im.size
        gts = []
        for line in gt_lbl.read_text().strip().split('\n'):
            parts = line.strip().split()
            if len(parts) != 5: continue
            c = int(parts[0]); cx, cy, w, h = map(float, parts[1:])
            gts.append((c, *yolo_to_xyxy(cx, cy, w, h, W, H)))
            n_gt_per_class[c] += 1
        used_gt = [False] * len(gts)

        # Match predictions tu cao -> thap conf
        order = np.argsort(-np.array(confs))
        for idx in order:
            pb, pc, pf = boxes[idx], classes[idx], confs[idx]
            if pc < 0:
                continue
            best_iou = 0; best_g = -1
            for gi, gt in enumerate(gts):
                if used_gt[gi] or gt[0] != pc:
                    continue
                io = iou(pb, gt[1:])
                if io > best_iou:
                    best_iou = io; best_g = gi
            tp = best_iou >= 0.5 and best_g >= 0
            if tp: used_gt[best_g] = True
            preds_per_class[pc].append((pf, tp))

    # Tinh AP per class (PR curve area)
    print(f'\n  {"class":<17} {"#GT":>6} {"#Pred":>6} {"AP@50":>8}')
    print(f'  {"-"*17} {"-"*6} {"-"*6} {"-"*8}')
    aps = []
    for c in range(8):
        preds = sorted(preds_per_class[c], key=lambda x: -x[0])
        n_gt = n_gt_per_class[c]
        if n_gt == 0:
            print(f'  {CLASS_NAMES[c]:<17} {0:>6} {len(preds):>6} {"N/A":>8}')
            continue
        tp = np.array([1 if p[1] else 0 for p in preds])
        fp = 1 - tp
        cum_tp = np.cumsum(tp); cum_fp = np.cumsum(fp)
        recall = cum_tp / max(n_gt, 1)
        precision = cum_tp / np.maximum(cum_tp + cum_fp, 1)
        # 11-point AP
        ap = 0.0
        for r in np.linspace(0, 1, 11):
            mask = recall >= r
            if mask.any():
                ap += precision[mask].max() / 11
        aps.append(ap)
        print(f'  {CLASS_NAMES[c]:<17} {n_gt:>6} {len(preds):>6} {ap:>8.4f}')
    print(f'\n  mAP@50 (combined pipeline): {np.mean(aps):.4f}')

## 6. So sanh per-class: Stage 2 standalone vs Combined

In [ ]:
# Stage 2 standalone tren test crops -> top-1 acc per class
from collections import defaultdict

model_s2_eval = YOLO(str(S2_BEST))
test_per_class = defaultdict(lambda: [0, 0])  # [correct, total]

for cname in CLASS_NAMES:
    cdir = STAGE2_DIR / 'test' / cname
    if not cdir.is_dir():
        continue
    crops = list(cdir.iterdir())
    if not crops:
        continue
    # Predict batch
    rs = model_s2_eval.predict(crops, imgsz=S2_IMGSZ, device=S2_DEVICE,
                               save=False, verbose=False)
    for r in rs:
        pred_idx = int(r.probs.top1)
        pred_name = S2_NAMES[pred_idx]
        test_per_class[cname][1] += 1
        if pred_name == cname:
            test_per_class[cname][0] += 1

print(f'  {"class":<17} {"Acc":>8} {"correct":>8} {"total":>8}')
print(f'  {"-"*17} {"-"*8} {"-"*8} {"-"*8}')
for cname in CLASS_NAMES:
    c, t = test_per_class[cname]
    acc = c / t if t else 0
    print(f'  {cname:<17} {acc:>8.4f} {c:>8} {t:>8}')

## 7. Visualize predictions tren test set

In [ ]:
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches

random.seed(13)
samples = random.sample(test_imgs, 6)

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
for ax, img_path in zip(axes.flat, samples):
    boxes, classes, confs = run_pipeline(img_path, conf_s1=0.20)
    with Image.open(img_path) as im:
        ax.imshow(im)
    cmap = plt.get_cmap('tab10')
    for b, c, f in zip(boxes, classes, confs):
        if c < 0 or f < 0.10: continue
        rect = patches.Rectangle((b[0], b[1]), b[2]-b[0], b[3]-b[1],
                                  linewidth=2, edgecolor=cmap(c), facecolor='none')
        ax.add_patch(rect)
        ax.text(b[0], b[1]-5, f'{CLASS_NAMES[c]} {f:.2f}',
                color='white', fontsize=8,
                bbox=dict(boxstyle='round,pad=0.3', fc=cmap(c), ec='none'))
    ax.set_title(img_path.name[:30], fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 8. Goi gon ket qua tai ve

In [ ]:
OUT_ZIP = WORK / f'results_2stage_{datetime.now().strftime("%m%d_%H%M")}.zip'
with zipfile.ZipFile(OUT_ZIP, 'w', zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for d in (Path(S1_PROJECT), Path(S2_PROJECT)):
        if not d.exists(): continue
        for root, _, files in os.walk(d):
            for f in files:
                full = Path(root) / f
                zf.write(full, full.relative_to(WORK))
print(f'Output: {OUT_ZIP} ({OUT_ZIP.stat().st_size/1e6:.1f} MB)')